In [5]:
# sqlite3: 파일 기반 데이터베이스(SQLite)를 사용할 수 있게 해주는 표준 라이브러리입니다.
# 여기서는 그래프의 상태(메모리)를 저장하기 위해 사용합니다.
import sqlite3

# LangGraph에서 상태 그래프를 만들기 위한 클래스와
# 그래프의 시작/끝을 나타내는 상수를 가져옵니다.
from langgraph.graph import StateGraph, START, END

# (주석 처리된) LangChain의 OpenAI 채팅 모델 초기화 함수입니다.
from langchain.chat_models import init_chat_model

# Google Cloud Vertex AI 의 Gemini 모델을 사용하기 위한 LangChain 래퍼입니다.
from langchain_google_vertexai import ChatVertexAI

# 대화 메시지(history)를 상태로 다룰 수 있게 해주는 기본 상태 타입입니다.
from langgraph.graph.message import MessagesState

# ToolNode, tools_condition:
# - ToolNode: LLM 이 "도구를 써야 한다"고 판단했을 때 실제로 도구를 실행해 주는 노드
# - tools_condition: LLM 응답 안에 도구 호출이 있는지 보고,
#                    다음에 ToolNode 로 갈지/그냥 넘어갈지 결정하는 조건 함수
from langgraph.prebuilt import ToolNode, tools_condition

# @tool 데코레이터로 파이썬 함수를 LLM 이 쓸 수 있는 "도구"로 등록할 때 사용합니다.
from langchain_core.tools import tool

# interrupt, Command:
# - interrupt: 그래프 실행을 잠시 멈추고, 사람(휴먼)의 입력을 기다리게 하는 기능입니다.
# - Command: 그래프를 다시 재개(resume)할 때 사용할 명령 객체입니다.
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt, Command

# OpenAI 모델을 쓰고 싶을 때는 아래 코드처럼 init_chat_model 을 사용할 수 있습니다.
# (지금은 Google Vertex AI 를 사용하므로 주석 처리되어 있습니다.)
# llm = init_chat_model("openai:gpt-4o-mini")

# Vertex AI 의 Gemini 모델을 LLM 으로 사용합니다.
llm = ChatVertexAI(
  model_name="gemini-2.5-flash-lite",
  project="ai-prompt-evaluator-489612",
  location="us-central1",
  max_tokens=500
)

# SQLite 데이터베이스 파일(memory-sync.db)에 연결합니다.
# - 이 파일 안에 그래프의 상태(체크포인트)가 저장되어,
#   나중에 중간 지점에서 다시 이어서 실행할 수 있습니다.
conn = sqlite3.connect(
  "memory-sync.db",
  check_same_thread=False,
)

# LangGraph 실행 시 사용할 기본 config 입니다.
# - configurable.thread_id: "3" 이라는 ID 로 이 대화/세션을 구분합니다.
config = {
  "configurable": {
    "thread_id": "3",
  }
}

In [6]:
# 이 그래프에서 사용할 상태(state)의 모양을 정의합니다.
# MessagesState 를 상속하면 기본적으로 "messages" 키에 대화 내역이 들어갑니다.
# 여기에 custom_stuff 같은 추가 필드를 정의해서, 필요하면 더 많은 정보를 담을 수 있습니다.
class State(MessagesState):
  custom_stuff: str

# 위에서 정의한 State 타입을 사용하는 상태 그래프를 만듭니다.
# 앞으로 노드들을 이 graph_builder 에 등록하고, 간선(실행 순서)을 정의합니다.
graph_builder = StateGraph(State)


In [7]:
# 이 도구는 "사람에게 시(poem)에 대한 피드백을 받아오는" 역할을 합니다.
# @tool 데코레이터를 사용해, LLM 이 이 함수를 도구(tool)로 호출할 수 있게 만듭니다.
@tool
def get_human_feedback(poem: str):
  """
  Asks the user for feedback on the poem.
  Use this before returning the final response.
  """
  # interrupt(...): 그래프 실행을 잠시 멈추고, 사람의 입력을 기다리게 합니다.
  # - LLM 이 작성한 시를 사람에게 보여주고,
  #   "어떻게 생각하는지"를 받아오도록 중간에 끼어드는(human-in-the-loop) 부분입니다.
  feedback = interrupt(f"Here is the poem, tell me what you think\n{poem}")

  # 사용자가 입력한 피드백을 그대로 반환합니다.
  return feedback

# LLM 에게 "너는 get_human_feedback 라는 도구를 쓸 수 있어" 라고 알려주는 단계입니다.
# 이렇게 하면 LLM 이 응답을 만들 때 이 도구를 호출하라는 요청을 생성할 수 있습니다.
llm_with_tools = llm.bind_tools(
  tools = [
    get_human_feedback,
  ]
)

# chatbot 노드는 "시를 잘 쓰는 전문가" 역할을 하는 LLM 을 한 번 호출하는 노드입니다.
# - 프롬프트 안에서: 반드시 get_human_feedback 도구를 먼저 써서
#   사람에게 피드백을 받고, 긍정적인 피드백을 받은 후에 최종 시를 내놓으라고 지시합니다.
# - state["messages"]: 지금까지의 대화 내역을 함께 전달합니다.
def chatbot(state: State):
  response = llm_with_tools.invoke(
    f"""
    You are an expert in making poems.
    Use the `get_human_feedback` tool to get feedback on your poem.
    Only after you receive positive feedback you can return the final poem.
    ALWAYS ASK FOR FEEDBACK FIRST.
    Here is the conversation history:
    {state["messages"]}
    """
  )

  # LangGraph 가 기존 messages 와 합칠 수 있도록,
  # 새로 받은 응답 메시지를 리스트에 담아 반환합니다.
  return {
    "messages": [response],
  }

In [8]:
# ToolNode 는 LLM 이 요청한 도구 호출을 실제로 실행해 주는 노드입니다.
# - 여기서는 get_human_feedback 도구 하나만 등록했습니다.
tool_node = ToolNode(
  tools = [
    get_human_feedback,
  ],
)

# 그래프에 노드들을 등록합니다.
# - "chatbot": 시를 생성하고, 필요하면 get_human_feedback 도구를 호출하라고 LLM 에게 지시하는 노드
# - "tools": 실제로 get_human_feedback 도구를 실행해서, 사람에게서 피드백을 받아오는 노드
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)

# 실행 흐름 정의
# 1) START -> chatbot: 먼저 챗봇 노드가 실행되어, LLM 이 시를 만들고 도구 호출을 요청할 수 있습니다.
# 2) chatbot 이후에는 tools_condition 에 따라 분기합니다.
#    - LLM 응답 안에 도구 호출(get_human_feedback)이 있으면 tools 노드로 이동
#    - 없으면 도구를 거치지 않고 종료할 수 있는 패턴입니다.
# 3) tools 노드에서 실제로 도구를 실행한 뒤, 다시 chatbot 으로 돌아와
#    사람 피드백을 반영한 최종 시를 완성할 수 있습니다.
graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

# 그래프를 컴파일하면서 SqliteSaver(checkpointer)를 연결합니다.
# - 이 설정 덕분에 그래프의 상태가 memory-sync.db 에 저장되고,
#   중간에 interrupt 로 멈춘 뒤 나중에 다시 이어서 실행(resume)할 수 있습니다.
graph = graph_builder.compile(
  checkpointer = SqliteSaver(conn),
)

In [9]:
# 그래프를 한 번 실행해 봅니다.
# - 초기 상태로 messages 에 사용자의 요청을 넣어 줍니다.
#   ("Python 코드에 대한 시를 써 주세요")
# - config 에 thread_id="3" 을 넘겨, 이 대화를 특정 세션으로 구분합니다.
result = graph.invoke(
  {
    "messages": [
      {
        "role": "user",
        "content": "Please make a poem about Python code."
      }
    ]
  },
  config=config
)

In [10]:
# 그래프 실행 결과로 받은 messages 를 예쁘게 출력해 봅니다.
# - 일반적으로: user 메시지 + LLM 의 첫 응답(도구 호출을 하려는 설명 등)이 들어 있습니다.
for message in result["messages"]:
  message.pretty_print()

================================ Human Message =================================

Please make a poem about Python code.
================================== Ai Message ==================================

I'd love to write a poem about Python code! But before I do, I need to get some feedback from you. Could you please tell me what you'd like in the poem? For example, are there any specific aspects of Python you'd like me to touch upon, like its readability, its versatility, or maybe a particular library?


In [11]:
# 현재 thread_id="3" 에 대한 그래프 상태(체크포인트)를 가져옵니다.
# - snapshot.next: "다음에 실행될 노드" 정보를 담고 있습니다.
#   interrupt 로 멈춘 뒤라면, 여기서 어디까지 진행되었는지 확인할 수 있습니다.
snapshot = graph.get_state(config)
snapshot.next

()

In [12]:
# 사람이(휴먼이) 시를 보고 "It looks great!" 이라고 피드백했다고 가정합니다.
# Command(resume=...) 를 사용하면, interrupt 로 멈췄던 그래프를
# 이 피드백을 가지고 다시 이어서 실행(resume)할 수 있습니다.
response = Command(resume = "It looks great!")

# 위에서 만든 Command 객체를 그래프에 넘겨, 이어서 실행합니다.
result = graph.invoke(
  response,
  config=config,
)

# 이제 LLM 은 "긍정적인 피드백을 받았다"고 판단하고,
# 실제 최종 시를 생성해서 messages 로 돌려줄 수 있습니다.
for message in result["messages"]:
  message.pretty_print

In [13]:
# 다시 한 번 현재 상태를 확인합니다.
# - 모든 흐름이 끝났다면, next 가 비어 있거나 종료 상태를 가리키게 됩니다.
snapshot = graph.get_state(config)
snapshot.next

()